# Nghi?m thu daily pipeline v? Data Quality t? ??ng

Notebook ki?m tra run `daily_20260101_20260722T011405`, tr?ng th?i ETL theo ??ng `pipeline_run_id`, k?t qu? 19 DQ checks v? d? li?u Silver/Gold sau khi ch?y ng?y `2026-01-01`. Logic nghi?m thu n?m trong Spark SQL; Python ch? k?t n?i PostgreSQL, hi?n th? v? d?ng khi c? `FAIL`.


In [1]:
import os

try:
    spark
except NameError as exc:
    raise RuntimeError("Hãy chọn kernel 'PySpark (Lakehouse)' trước khi chạy") from exc

PIPELINE_RUN_ID = "daily_20260101_20260722T011405"
COB_DT = "2026-01-01"
jdbc_url = (
    f"jdbc:postgresql://{os.environ.get('POSTGRES_HOST', 'postgres')}:"
    f"{os.environ.get('POSTGRES_PORT', '5432')}/{os.environ['POSTGRES_DB']}"
)

def read_postgres(query):
    return (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("query", query)
        .option("user", os.environ["POSTGRES_USER"])
        .option("password", os.environ["POSTGRES_PASSWORD"])
        .option("driver", "org.postgresql.Driver")
        .load()
    )

## 1. Run-aware ETL flags

Mỗi child DAG phải có trạng thái mới nhất là `S` trong đúng business date và đúng pipeline run; success cũ không được dùng thay thế.

In [2]:
flags = read_postgres(f"""
    SELECT id, job_name, status, cob_dt, dag_run_id, pipeline_run_id, try_number
    FROM opslakehouse.flag_job_etl
    WHERE pipeline_run_id = '{PIPELINE_RUN_ID}' AND cob_dt = DATE '{COB_DT}'
""")
flags.createOrReplaceTempView("acceptance_flags")
flag_contract = spark.sql("""
WITH expected(job_name) AS (
    VALUES ('bronze_core_banking_dag'), ('bronze_card_crm_dag'),
           ('silver_all_dag'), ('gold_mart360_dag'),
           ('gold_time_analytics_dag'), ('gold_segmentation_dag'),
           ('ops_pii_masking_daily_dag'), ('ops_dq_daily_dag')
), ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY job_name ORDER BY id DESC) AS rn
    FROM acceptance_flags
), latest AS (
    SELECT job_name, status, dag_run_id FROM ranked WHERE rn = 1
)
SELECT COALESCE(e.job_name, l.job_name) AS job_name, l.status, l.dag_run_id,
       CASE WHEN e.job_name IS NOT NULL AND l.status = 'S' THEN 'PASS' ELSE 'FAIL' END AS result
FROM expected e FULL OUTER JOIN latest l ON e.job_name = l.job_name
ORDER BY job_name
""")
flag_contract.show(20, truncate=False)
assert flag_contract.where("result = 'FAIL'").count() == 0, "Run-aware ETL flag contract failed"

+-------------------------+------+--------------------------------------------------------+------+
|job_name                 |status|dag_run_id                                              |result|
+-------------------------+------+--------------------------------------------------------+------+
|bronze_card_crm_dag      |S     |manual__daily_20260101_20260722T011405_bronze_card      |PASS  |
|bronze_core_banking_dag  |S     |manual__daily_20260101_20260722T011405_bronze_core      |PASS  |
|gold_mart360_dag         |S     |manual__daily_20260101_20260722T011405_gold_mart360     |PASS  |
|gold_segmentation_dag    |S     |manual__daily_20260101_20260722T011405_gold_segmentation|PASS  |
|gold_time_analytics_dag  |S     |manual__daily_20260101_20260722T011405_gold_time        |PASS  |
|ops_dq_daily_dag         |S     |manual__daily_20260101_20260722T011405_dq               |PASS  |
|ops_pii_masking_daily_dag|S     |manual__daily_20260101_20260722T011405_masking          |PASS  |
|silver_al

## 2. Kết quả DQ đã được lưu trong PostgreSQL

Critical failure làm DAG thất bại; warning được lưu để quan sát nhưng không chặn pipeline.

In [3]:
dq_results = read_postgres(f"""
    SELECT check_name, severity, check_type, passed, failed_count, total_count, error_detail
    FROM opslakehouse.dq_check_result
    WHERE pipeline_run_id = '{PIPELINE_RUN_ID}' AND cob_dt = DATE '{COB_DT}'
    ORDER BY severity, check_name
""")
dq_results.createOrReplaceTempView("acceptance_dq_results")
dq_results.show(30, truncate=False)
dq_contract = spark.sql("""
SELECT COUNT(*) AS check_count,
       SUM(CASE WHEN severity = 'critical' AND NOT passed THEN 1 ELSE 0 END) AS critical_failures,
       SUM(CASE WHEN error_detail IS NOT NULL AND failed_count = -1 THEN 1 ELSE 0 END) AS execution_errors,
       CASE WHEN COUNT(*) = 19
                  AND SUM(CASE WHEN severity = 'critical' AND NOT passed THEN 1 ELSE 0 END) = 0
                  AND SUM(CASE WHEN error_detail IS NOT NULL AND failed_count = -1 THEN 1 ELSE 0 END) = 0
            THEN 'PASS' ELSE 'FAIL' END AS status
FROM acceptance_dq_results
""")
dq_contract.show(truncate=False)
assert dq_contract.where("status = 'FAIL'").count() == 0, "Automated DQ contract failed"

+-----------------------------------------+--------+---------------------+------+------------+-----------+------------+
|check_name                               |severity|check_type           |passed|failed_count|total_count|error_detail|
+-----------------------------------------+--------+---------------------+------+------------+-----------+------------+
|aum_reconciles_to_independent_sources    |critical|reconciliation       |true  |0           |10000      |NULL        |
|business_flags_are_binary                |critical|range                |true  |0           |30000      |NULL        |
|campaign_unique_customer_date            |critical|uniqueness           |true  |0           |10000      |NULL        |
|dashboard_customer_date_is_unique        |critical|uniqueness           |true  |0           |10000      |NULL        |
|dashboard_has_no_raw_pii_columns         |critical|schema_columns_absent|true  |0           |46         |NULL        |
|dashboard_nbo_contract_is_valid        

## 3. Daily facts không xóa lịch sử và không tạo orphan SK

In [4]:
fact_contract = spark.sql("""
SELECT table_name, row_count, business_key_count, orphan_count,
       CASE WHEN row_count = business_key_count AND orphan_count = 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM (
    SELECT 'fact_txn_account' AS table_name, COUNT(*) AS row_count,
           COUNT(DISTINCT txn_id) AS business_key_count,
           SUM(CASE WHEN account_sk IS NULL OR customer_sk IS NULL THEN 1 ELSE 0 END) AS orphan_count
    FROM lakehouse.silver.fact_txn_account
    UNION ALL
    SELECT 'fact_card_txn', COUNT(*), COUNT(DISTINCT txn_id),
           SUM(CASE WHEN customer_sk IS NULL THEN 1 ELSE 0 END)
    FROM lakehouse.silver.fact_card_txn
    UNION ALL
    SELECT 'fact_crm_interaction', COUNT(*), COUNT(DISTINCT interaction_id),
           SUM(CASE WHEN customer_sk IS NULL THEN 1 ELSE 0 END)
    FROM lakehouse.silver.fact_crm_interaction
) facts
ORDER BY table_name
""")
fact_contract.show(truncate=False)
assert fact_contract.where("status = 'FAIL'").count() == 0, "Daily fact contract failed"

+--------------------+---------+------------------+------------+------+
|table_name          |row_count|business_key_count|orphan_count|status|
+--------------------+---------+------------------+------------+------+
|fact_card_txn       |212400   |212400            |0           |PASS  |
|fact_crm_interaction|10000    |10000             |0           |PASS  |
|fact_txn_account    |1050000  |1050000           |0           |PASS  |
+--------------------+---------+------------------+------------+------+



## 4. Serving và marketing cùng một snapshot

In [5]:
serving_contract = spark.sql(f"""
WITH expected AS (
    SELECT COUNT(*) AS n FROM lakehouse.gold.mart_customer_360
    WHERE cob_dt = DATE '{COB_DT}'
), populations AS (
    SELECT 'mart_customer_360' AS table_name, COUNT(*) AS n
    FROM lakehouse.gold.mart_customer_360 WHERE cob_dt = DATE '{COB_DT}'
    UNION ALL SELECT 'mart_customer_360_masked', COUNT(*)
    FROM lakehouse.sandbox.mart_customer_360_masked WHERE cob_dt = DATE '{COB_DT}'
    UNION ALL SELECT 'rfm_segment', COUNT(*)
    FROM lakehouse.gold.rfm_segment WHERE cob_dt = DATE '{COB_DT}'
    UNION ALL SELECT 'churn_prediction', COUNT(*)
    FROM lakehouse.gold.churn_prediction WHERE cob_dt = DATE '{COB_DT}'
    UNION ALL SELECT 'cross_sell_segment', COUNT(*)
    FROM lakehouse.gold.cross_sell_segment WHERE cob_dt = DATE '{COB_DT}'
    UNION ALL SELECT 'campaign_target', COUNT(*)
    FROM lakehouse.gold.campaign_target WHERE cob_dt = DATE '{COB_DT}'
)
SELECT p.table_name, p.n AS row_count, e.n AS expected_count,
       CASE WHEN p.n = e.n AND e.n > 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM populations p CROSS JOIN expected e
ORDER BY p.table_name
""")
serving_contract.show(truncate=False)
assert serving_contract.where("status = 'FAIL'").count() == 0, "Serving population contract failed"

+------------------------+---------+--------------+------+
|table_name              |row_count|expected_count|status|
+------------------------+---------+--------------+------+
|campaign_target         |10000    |10000         |PASS  |
|churn_prediction        |10000    |10000         |PASS  |
|cross_sell_segment      |10000    |10000         |PASS  |
|mart_customer_360       |10000    |10000         |PASS  |
|mart_customer_360_masked|10000    |10000         |PASS  |
|rfm_segment             |10000    |10000         |PASS  |
+------------------------+---------+--------------+------+



In [6]:
print("DAILY PIPELINE + DQ ACCEPTANCE: PASS")

DAILY PIPELINE + DQ ACCEPTANCE: PASS
